# Version 2: Unsloth Active Agentic Chatbot
This notebook trains an 'Active Agentic Chatbot' using the `Llama-3.2-3B-Instruct` model. Multi-Task Fine-Tuning is utilized to teach the model to chat as a doctor, and output a SOAP note based on the chat history when triggered. 

Hardware constraints (RTX 3060 with 6GB VRAM) have necessitated the use of the `unsloth` library for 4-bit NF4 quantization and memory patching to prevent Out Of Memory (OOM) errors.

## Step 1: Install the Unsloth Environment
We are skipping standard `transformers` and `peft` packages in favor of `unsloth`'s custom memory-patched wheels to avoid VRAM crashes.

## Step 2: Load the Model (Memory Strict)
Initialize the `Llama-3.2-3B-Instruct` model using Unsloth�s `FastLanguageModel`. This auto-applies exactly what we need: 4-bit quantization and patched inference mechanisms.

In [2]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Llama-3.2-3B-Instruct"

# Using max_seq_length=1024 caps our context window to conservatively manage exactly around 6GB VRAM
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 4096,
    dtype = None, # Auto-detects bfloat16 for RTX 30-series
    load_in_4bit = True,
)

print("FastLanguageModel loaded in 4-bit successfully.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


FastLanguageModel loaded in 4-bit successfully.


## Step 3: Inject Unsloth QLoRA Adapters
Inject LoRA adapters specifically into attention and MLP layers. We use `use_gradient_checkpointing="unsloth"` to dramatically cut backprop memory.

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Unsloth optimizes dropout to 0 for a 20% speedup
    bias = "none",
    use_gradient_checkpointing = "unsloth", # CRITICAL for 6GB VRAM
)

print("Unsloth PEFT QLoRA Adapters successfully injected.")


Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth PEFT QLoRA Adapters successfully injected.


## Step 4: The Multi-Task Data Mapper (The Secret Sauce)
Map the `omi-health/medical-dialogue-to-soap-summary` into an interactive Turn-By-Turn Chat Template. A final "Generate clinical documentation." user trigger acts as the bridge that demands a SOAP summarization constraint output.

In [4]:
import re
from datasets import load_dataset

def format_agentic_chat(example):
    system_prompt = "You are an empathetic, professional AI medical assistant. You will chat with the patient to understand their symptoms. When the user says 'Generate clinical documentation', you will instantly switch modes and output a highly structured SOAP note based ONLY on the conversation history."
    messages = [{"role": "system", "content": system_prompt}]
    
    # 1. Parse the dialogue into back-and-forth chat turns
    # The dataset uses "Doctor: " and "Patient: " prefixes
    turns = re.split(r'(Doctor:|Patient:)', example.get('dialogue', example.get('Dialogue', '')))
    
    current_role = None
    current_text = ""
    
    for part in turns:
        part = part.strip()
        if not part: continue
        
        if part == "Doctor:":
            if current_role == "user" and current_text:
                messages.append({"role": "user", "content": current_text.strip()})
            current_role = "assistant"
            current_text = ""
        elif part == "Patient:":
            if current_role == "assistant" and current_text:
                messages.append({"role": "assistant", "content": current_text.strip()})
            current_role = "user"
            current_text = ""
        else:
            current_text += " " + part

    # Append the final spoken turn
    if current_role and current_text:
         messages.append({"role": current_role, "content": current_text.strip()})
         
    # 2. Add the Multi-Task Trigger for Summarization
    messages.append({"role": "user", "content": "Generate clinical documentation."})
    
    # 3. Add the Expected Summary
    messages.append({"role": "assistant", "content": example.get("summary", example.get("soap", ""))}) # The SOAP note target
    
    # 4. Apply the Llama 3.2 template (add_generation_prompt=False during training!)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# Load and format the dataset
dataset = load_dataset("omi-health/medical-dialogue-to-soap-summary", split="train")

print("Formatting dataset for multi-task prompt chaining...")
dataset = dataset.map(format_agentic_chat)
print(f"Sample zero text block:\n{dataset[0]['text'][:500]}...")


Formatting dataset for multi-task prompt chaining...
Sample zero text block:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 11 Mar 2026

You are an empathetic, professional AI medical assistant. You will chat with the patient to understand their symptoms. When the user says 'Generate clinical documentation', you will instantly switch modes and output a highly structured SOAP note based ONLY on the conversation history.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hello, how can I help you today?<|...


## Step 5: Generate the DPO Preference Dataset (For Version 3)
Extracting the baseline errors from Version 1's `v1_llama_baseline.csv`. These hallucinated generations will form the "rejected" responses, while the dataset Ground Truths act as the "chosen" responses for future alignment.

In [ ]:
import pandas as pd
import json

# Read the baseline CSV generated in Version 1
try:
    df_baseline = pd.read_csv("../data/v1_llama_baseline.csv")
    
    dpo_data = []
    for _, row in df_baseline.iterrows():
        # Setup context and preference mappings
        dpo_data.append({
            "prompt": row["Dialogue"],
            "chosen": row["Ground Truth (SOAP)"],
            "rejected": row["Llama_Prediction"]
        })
        
    with open("v3_dpo_preference_data.json", "w") as f:
        json.dump(dpo_data, f, indent=4)
        
    print(f"Successfully generated 'v3_dpo_preference_data.json' with {len(dpo_data)} preference pairs.")
except FileNotFoundError:
    print("Warning: 'v1_llama_baseline.csv' not found! Make sure Version 1 was successfully executed in the current directory.")


Successfully generated 'v3_dpo_preference_data.json' with 49 preference pairs.


## Step 6: Configure SFTTrainer for RTX 3060
Use the `.trl` extension `SFTTrainer` with targeted strict memory configurations, notably restricting `gradient_accumulation_steps=4` with `per_device_train_batch_size=2` for stability.

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 16, # Simulates a batch size of 8
        warmup_steps = 5,
        num_train_epochs = 1, # Trains on the entire dataset exactly once
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_torch", # Offloads optimizer states to RAM
        output_dir = "agentic_medsoap_outputs",
    ),
)

print("Beginning SFT Training loop!")
trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=54):   0%|          | 0/9250 [00:00<?, ? examples/s]

Beginning SFT Training loop!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,250 | Num Epochs = 1 | Total steps = 290
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss
1,2.088202
2,2.088783
3,2.033835
4,1.910233
5,1.661387
6,1.608783
7,1.548933
8,1.445919
9,1.382102
10,1.334848


TrainOutput(global_step=290, training_loss=1.0489966497339052, metrics={'train_runtime': 1171.3501, 'train_samples_per_second': 7.897, 'train_steps_per_second': 0.248, 'total_flos': 1.639110143857705e+17, 'train_loss': 1.0489966497339052, 'epoch': 1.0})

## Step 7: Save the Fine-Tuned Adapters
We export specific adapter weights separately locally utilizing `save_pretrained` to save footprint over standard monolithic checkpoint logic.

In [ ]:
# Extracting merely the highly compressed Adapter weights
model.save_pretrained("v2_agentic_adapters")
tokenizer.save_pretrained("v2_agentic_adapters")

print("Safely persisted local adapter delta matrices to `v2_agentic_adapters`\nTraining Run completed!")


In [1]:
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps xformers trl peft accelerate bitsandbytes datasets hf_transfer


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-e6hfsxxh/unsloth_09c3f143e0bb4bf39851f4604518e8f9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-e6hfsxxh/unsloth_09c3f143e0bb4bf39851f4604518e8f9
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
